In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

DATA_DIR = "D:/Machine-Learning/HousePrice prediction/data"
def read_data():
    return pd.read_csv(f"{DATA_DIR}/Housing.csv")
df = read_data()


In [2]:
df.isnull().sum()


price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

In [3]:
# encode categorical data with binary encoding and multiclass with one-hot encoding
def encode_categorical(df):
    df = df.copy()

    binary_cols = [
        'mainroad',
        'guestroom',
        'basement',
        'hotwaterheating',
        'airconditioning',
        'prefarea'
    ]

    for col in binary_cols:
        df[col] = df[col].map({'yes': 1, 'no': 0})

    # multiclass column hot one encoding
    # ir covertts categorical text data into numreical columns
    df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=False, dtype=int)
    return df


In [4]:
def clean_outliers(df, column):
    df = df.copy()
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    df[column] = np.where(df[column] < lower_bound, lower_bound,
                          np.where(df[column] > upper_bound, upper_bound, df[column]))
    return df


In [5]:
df_encoded = encode_categorical(df)

# remove outliers
df_encoded = clean_outliers(df_encoded, 'area')
df_encoded = clean_outliers(df_encoded, 'price')

df_encoded.info()


<class 'pandas.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   price                            545 non-null    float64
 1   area                             545 non-null    float64
 2   bedrooms                         545 non-null    int64  
 3   bathrooms                        545 non-null    int64  
 4   stories                          545 non-null    int64  
 5   mainroad                         545 non-null    int64  
 6   guestroom                        545 non-null    int64  
 7   basement                         545 non-null    int64  
 8   hotwaterheating                  545 non-null    int64  
 9   airconditioning                  545 non-null    int64  
 10  parking                          545 non-null    int64  
 11  prefarea                         545 non-null    int64  
 12  furnishingstatus_furnished       

In [6]:
# creating a new column : price per area
def price_per_area(df):
    df = df.copy()
    df['price_per_area'] = df['price'] / df['area']
    return df


In [7]:
df_encoded = price_per_area(df_encoded)


In [8]:
# split data into train and test sets
# split before scaling to prevent data leakage, as scaling involves using mean and std
x = df_encoded.drop('price', axis=1)
y = df_encoded['price']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [9]:
# feature scaling (normalizing)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

scaler.fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)
import pandas as pd

x_train_scaled = pd.DataFrame(
    x_train_scaled,
    columns=x_train.columns,
    index=x_train.index
)

x_test_scaled = pd.DataFrame(
    x_test_scaled,
    columns=x_test.columns,
    index=x_test.index
)
x_train_scaled.to_csv('x_train_scaled.csv', index=False)
x_test_scaled.to_csv('x_test_scaled.csv', index=False)

y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)


In [10]:
# check final shape, no nulls and all numeric
print("Shape of dataset:", df_encoded.shape)


Shape of dataset: (545, 16)


In [11]:
print("Total missing values:", df_encoded.isnull().sum().sum())


Total missing values: 0


In [12]:
print(df_encoded.dtypes)


price                              float64
area                               float64
bedrooms                             int64
bathrooms                            int64
stories                              int64
mainroad                             int64
guestroom                            int64
basement                             int64
hotwaterheating                      int64
airconditioning                      int64
parking                              int64
prefarea                             int64
furnishingstatus_furnished           int64
furnishingstatus_semi-furnished      int64
furnishingstatus_unfurnished         int64
price_per_area                     float64
dtype: object
